In [1]:
import sys
import os

# Add the project root to Python's search path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now import the dataframes
from scripts.data_loading import consumers, accounts, transactions, category_mapping

In [2]:
# individual consumer data
consumers.head()

,prism_consumer_id,evaluation_date,credit_score,DQ_TARGET
0,0,2021-09-01,726.0,0.0
1,1,2021-07-01,626.0,0.0
2,2,2021-05-01,680.0,0.0
3,3,2021-03-01,734.0,0.0
4,4,2021-10-01,676.0,0.0


In [3]:
# account information
accounts.head()

,prism_consumer_id,prism_account_id,account_type,balance_date,balance
0,3023,0,SAVINGS,2021-08-31,90.57
1,3023,1,CHECKING,2021-08-31,225.95
2,4416,2,SAVINGS,2022-03-31,15157.17
3,4416,3,CHECKING,2022-03-31,66.42
4,4227,4,CHECKING,2021-07-31,7042.90


In [4]:
# transactional data
transactions.head()

,prism_consumer_id,prism_transaction_id,category,amount,credit_or_debit,posted_date
0,3023,0,4,0.05,CREDIT,2021-04-16
1,3023,1,12,481.56,CREDIT,2021-04-30
2,3023,2,4,0.05,CREDIT,2021-05-16
3,3023,3,4,0.07,CREDIT,2021-06-16
4,3023,4,4,0.06,CREDIT,2021-07-16


In [5]:
# category mapping
category_mapping

,category_id,category
0,0,SELF_TRANSFER
1,1,EXTERNAL_TRANSFER
2,2,DEPOSIT
3,3,PAYCHECK
4,4,MISCELLANEOUS
5,5,PAYCHECK_PLACEHOLDER
6,6,REFUND
7,7,INVESTMENT_INCOME
8,8,OTHER_BENEFITS
9,9,UNEMPLOYMENT_BENEFITS


In [6]:
import pandas as pd

accounts['balance_date'] = pd.to_datetime(accounts['balance_date'])
transactions['posted_date'] = pd.to_datetime(transactions['posted_date'])

In [7]:
# Check duplication across the four datasets
from collections import Counter

datasets = {
    'consumers': consumers,
    'accounts': accounts,
    'transactions': transactions,
    'category_mapping': category_mapping,
}

for name, df in datasets.items():
    print('\n' + '='*80)
    print(f'Dataset: {name} — rows: {len(df):,}, columns: {df.shape[1]}')

    # Full-row duplicates
    dup_any = df.duplicated(keep=False).sum()
    dup_prev = df.duplicated(keep='first').sum()
    unique_rows = df.drop_duplicates().shape[0]
    print(f'Full-row duplicates (any in group): {dup_any:,}')
    print(f'Full-row duplicates (excluding first in group): {dup_prev:,}')
    print(f'Unique rows after dropping duplicates: {unique_rows:,}')

    # Look for common ID columns and report duplicate IDs
    id_cols = [c for c in ['prism_consumer_id', 'prism_account_id', 'prism_transaction_id'] if c in df.columns]
    if id_cols:
        for col in id_cols:
            n_missing = df[col].isna().sum()
            n_dup_ids = df[col].duplicated().sum()
            n_unique_ids = df[col].nunique(dropna=True)
            print(f"\nColumn: {col} — missing: {n_missing:,}, duplicate id rows: {n_dup_ids:,}, unique ids: {n_unique_ids:,}")
            if n_dup_ids > 0:
                # show top duplicated ids
                top_dups = df[col].value_counts().head(5)
                print('Top duplicated ids (count):\n', top_dups.to_string())
    else:
        # For mapping tables, check duplicates on all columns
        if name == 'category_mapping':
            dup_map = df.duplicated(subset=df.columns.tolist(), keep=False).sum()
            print(f"Category mapping duplicate rows (any in group): {dup_map:,}")

    # Show a few example duplicate rows (full-row duplicates)
    if dup_any > 0:
        print('\nExample duplicate rows:')
        display(df[df.duplicated(keep=False)].head(10))

    print('\n' + '='*80)

print('\nDuplicate check complete.')



Dataset: consumers — rows: 15,000, columns: 4
Full-row duplicates (any in group): 0
Full-row duplicates (excluding first in group): 0
Unique rows after dropping duplicates: 15,000

Column: prism_consumer_id — missing: 0, duplicate id rows: 0, unique ids: 15,000


Dataset: accounts — rows: 24,466, columns: 5
Full-row duplicates (any in group): 0
Full-row duplicates (excluding first in group): 0
Unique rows after dropping duplicates: 24,466

Column: prism_consumer_id — missing: 0, duplicate id rows: 11,457, unique ids: 13,009
Top duplicated ids (count):
 prism_consumer_id
14525    25
13491    14
14294    13
5109     12
7074     12

Column: prism_account_id — missing: 0, duplicate id rows: 0, unique ids: 24,466


Dataset: transactions — rows: 6,407,321, columns: 6
Full-row duplicates (any in group): 4,024
Full-row duplicates (excluding first in group): 2,012
Unique rows after dropping duplicates: 6,405,309

Column: prism_consumer_id — missing: 0, duplicate id rows: 6,392,829, unique ids:

,prism_consumer_id,prism_transaction_id,category,amount,credit_or_debit,posted_date
66482,8,66482,4,0.09,CREDIT,2021-08-16
66483,8,66482,4,0.09,CREDIT,2021-08-16
66484,8,66483,4,0.09,CREDIT,2021-09-16
66485,8,66483,4,0.09,CREDIT,2021-09-16
66486,8,66484,0,40.00,CREDIT,2021-09-21
66487,8,66484,0,40.00,CREDIT,2021-09-21
66488,8,66485,12,180.59,CREDIT,2021-09-22
66489,8,66485,12,180.59,CREDIT,2021-09-22
66490,8,66486,0,25.00,CREDIT,2021-10-06
66491,8,66486,0,25.00,CREDIT,2021-10-06




Dataset: category_mapping — rows: 50, columns: 2
Full-row duplicates (any in group): 0
Full-row duplicates (excluding first in group): 0
Unique rows after dropping duplicates: 50
Category mapping duplicate rows (any in group): 0


Duplicate check complete.


In [8]:
# Remove duplicate rows from `transactions` (prefer dedupe by transaction id when available)
print('\nRemoving duplicates from `transactions` dataframe...')

before = len(transactions)
if 'prism_transaction_id' in transactions.columns:
    dup_count = transactions['prism_transaction_id'].duplicated().sum()
    print(f'Found {dup_count:,} duplicate prism_transaction_id rows (including first occurrences).')
    transactions = (
        transactions.drop_duplicates(subset=['prism_transaction_id'], keep='first')
                    .reset_index(drop=True)
    )
    removed = before - len(transactions)
    print(f'Removed {removed:,} rows based on `prism_transaction_id` deduplication.')
else:
    dup_count = transactions.duplicated().sum()
    print(f'Found {dup_count:,} full-row duplicate rows.')
    transactions = transactions.drop_duplicates(keep='first').reset_index(drop=True)
    print(f'Removed {dup_count:,} full-row duplicates.')

# As an extra precaution, also remove exact duplicate rows (if any remain)
extra_dups = transactions.duplicated().sum()
if extra_dups > 0:
    transactions = transactions.drop_duplicates(keep='first').reset_index(drop=True)
    print(f'Also removed {extra_dups:,} additional exact duplicate rows.')

print(f'`transactions` now has {len(transactions):,} rows (was {before:,}).')

# Optionally show a quick sanity sample
print('\nSample rows after deduplication:')
print(transactions.head(3).to_string())



Removing duplicates from `transactions` dataframe...
Found 2,012 duplicate prism_transaction_id rows (including first occurrences).
Removed 2,012 rows based on `prism_transaction_id` deduplication.
`transactions` now has 6,405,309 rows (was 6,407,321).

Sample rows after deduplication:
  prism_consumer_id prism_transaction_id  category  amount credit_or_debit posted_date
0              3023                    0         4    0.05          CREDIT  2021-04-16
1              3023                    1        12  481.56          CREDIT  2021-04-30
2              3023                    2         4    0.05          CREDIT  2021-05-16
